# caliper quickstart

Run `caliper` on a Colab GPU runtime with no local setup. ~2 minutes for the
first cell (it builds the Rust extension from source; a released `pip install
caliper-gpu` will skip that).


In [ ]:
# 1. install from source (Rust toolchain + the package)
!curl -sSf https://sh.rustup.rs | sh -s -- -y >/dev/null && . $HOME/.cargo/env \
  && pip -q install 'git+https://github.com/mansoor-mamnoon/caliper'
!caliper --version


## Is this box fit to benchmark?


In [ ]:
!caliper doctor


## Measure a kernel

The live launcher is still a stub, so `bench` replays a recorded device
session. This fixture ships with the repo; a real run on this GPU is one flag
away once the launcher lands.


In [ ]:
!curl -sSL -o session.jsonl https://raw.githubusercontent.com/mansoor-mamnoon/caliper/main/crates/caliper-gpu/fixtures/bench/happy.jsonl
# the committed fixtures are recorded at batches=40
!caliper bench k --recording session.jsonl --batches 40 --json | python -m json.tool | head -40


## The reference corpus (pure, no GPU needed for the roofline)


In [ ]:
from caliper.corpus.kernels import gemm
print(gemm.roofline_spec({'M': 4096, 'N': 4096, 'K': 4096}, 'bf16'))
# gemm.run({'shape': {'m': 4096, 'n': 4096, 'k': 4096}, 'dtype': 'bf16', 'layout': 'row'})  # needs triton + CUDA


## Next

- [`docs/api.md`](https://github.com/mansoor-mamnoon/caliper/blob/main/docs/api.md) · [`docs/cli.md`](https://github.com/mansoor-mamnoon/caliper/blob/main/docs/cli.md)
- [`docs/why-do_bench-misleads.md`](https://github.com/mansoor-mamnoon/caliper/blob/main/docs/why-do_bench-misleads.md) — what a default `do_bench` gets wrong
- **Submit your GPU's numbers**: `caliper sweep` a corpus spec, then `caliper submit --out bundle/` and open a PR to `caliper-results`.
